# Fase 0 — Base do ambiente (lamp-sim)

Projeto LAMP · frente de Simulação, dados sintéticos e clusterização temporal.

Este notebook prepara e confere o ambiente no Google Colab:

1. conecta o Google Drive e encontra o código em `Meu Drive/LAMP/lamp-sim`;
2. cria as pastas de dados em `Meu Drive/LAMP/lamp-sim-dados` (fora do código, para não ir para o git);
3. roda a verificação de estilo e os testes automáticos;
4. registra versões e data num arquivo de log no Drive.

**Antes de rodar:** a pasta `lamp-sim` precisa estar em `Meu Drive/LAMP/`. Rode as células em ordem (*Ambiente de execução → Executar tudo*). Não precisa de GPU.

## 1. Conectar o Drive e localizar o código

In [ ]:
import sys
from pathlib import Path

EM_COLAB = "google.colab" in sys.modules
if EM_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/LAMP")
    CODIGO = BASE / "lamp-sim"
else:  # rodando fora do Colab (ex.: no computador)
    CODIGO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    BASE = CODIGO / "_dados_local"
DADOS = BASE / "lamp-sim-dados"   # fora do repositório: dados não vão para o git

assert (CODIGO / "src" / "lampsim").is_dir(), (
    f"Código não encontrado em {CODIGO}. Confira se a pasta 'lamp-sim' está em 'Meu Drive/LAMP'.")
if str(CODIGO / "src") not in sys.path:
    sys.path.insert(0, str(CODIGO / "src"))
print("Colab:", EM_COLAB)
print("Código:", CODIGO)
print("Dados :", DADOS)

## 2. Criar as pastas de dados

In [ ]:
PASTAS = {nome: DADOS / nome for nome in ("lotes", "resultados", "logs")}
for pasta in PASTAS.values():
    pasta.mkdir(parents=True, exist_ok=True)
    print("ok:", pasta)

## 3. Ferramentas de teste e testes automáticos

Os mesmos comandos da esteira automática do repositório (`.github/workflows/ci.yml`).

In [ ]:
import importlib.util
import subprocess

faltando = [m for m in ("pytest", "ruff") if importlib.util.find_spec(m) is None]
if faltando:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltando], check=True)
print("ferramentas ok")

In [ ]:
def rodar(*cmd):
    r = subprocess.run([sys.executable, "-m", *cmd], cwd=CODIGO, capture_output=True, text=True)
    print(r.stdout[-3000:] or r.stderr[-3000:])
    return r.returncode

estilo = rodar("ruff", "check", ".", "--no-cache")
testes = rodar("pytest", "-q", "-p", "no:cacheprovider")
assert estilo == 0, "Verificação de estilo falhou (veja acima)."
assert testes == 0, "Algum teste falhou (veja acima)."
print("Estilo e testes: tudo verde.")

## 4. Registrar o ambiente

Versões, data e configuração ficam num arquivo no Drive, para cada experimento poder ser refeito depois (exigência do protocolo LAMP-S02-PROT).

In [ ]:
import json
import platform
from datetime import datetime, timezone

import matplotlib
import numpy as np

import lampsim
from lampsim.config import VERSAO_PARAMETROS

agora = datetime.now(timezone.utc)
ambiente = {
    "data_utc": agora.isoformat(timespec="seconds"),
    "colab": EM_COLAB,
    "python": platform.python_version(),
    "numpy": np.__version__,
    "matplotlib": matplotlib.__version__,
    "lampsim": lampsim.__version__,
    "versao_parametros": VERSAO_PARAMETROS,
    "testes": "ok",
}
log = PASTAS["logs"] / f"ambiente_{agora:%Y%m%d_%H%M%S}.json"
log.write_text(json.dumps(ambiente, indent=2), encoding="utf-8")

# critério de pronto: o arquivo foi gravado e pode ser lido de volta
assert json.loads(log.read_text(encoding="utf-8")) == ambiente
print("Registro gravado em:", log)
print(json.dumps(ambiente, indent=2))

## Fase 0 pronta quando

- [ ] Este notebook roda do início ao fim numa sessão nova do Colab, sem erro.
- [ ] Estilo e testes aparecem como "tudo verde".
- [ ] Existe um arquivo `ambiente_*.json` em `Meu Drive/LAMP/lamp-sim-dados/logs`.

Próximo passo: `01_simulador.ipynb`.